In [2]:
import pandas as pd
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

# --- 1. CONFIGURATION ---
CSV_FILE = 'healthy_autoenc.csv' # Change this to the exact name of your CSV
WINDOW_SIZE = 100               # 100Hz = 1 second windows
NUM_AXES = 3                    # X, Y, Z
INPUT_SHAPE = WINDOW_SIZE * NUM_AXES # 300 flat floats

print("--- 1. LOADING DATA ---")
# Load CSV, assuming columns are ax, ay, az
df = pd.read_csv(CSV_FILE)
raw_data = df[['ax', 'ay', 'az']].values

# Chop the continuous data into 1-second chunks (100 samples each)
# Then flatten each 100x3 chunk into a single 300-length array
num_windows = len(raw_data) // WINDOW_SIZE
windows = raw_data[:num_windows * WINDOW_SIZE].reshape((num_windows, INPUT_SHAPE))

# Convert to float32 (required for TFLite)
X_train = windows.astype(np.float32)
print(f"Generated {X_train.shape[0]} training windows of shape {X_train.shape[1]}")

print("\n--- 2. BUILDING THE AUTOENCODER ---")
model = tf.keras.Sequential([
    # Encoder
    tf.keras.layers.Dense(64, activation='relu', input_shape=(INPUT_SHAPE,)),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(16, activation='relu'), # The "Latent Space" bottleneck

    # Decoder
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(INPUT_SHAPE, activation='linear') # Reconstruct the 300 floats
])

model.compile(optimizer='adam', loss='mse')
model.summary()

print("\n--- 3. TRAINING THE AI ---")
# We use X_train for both input AND output because it is learning to copy itself
history = model.fit(X_train, X_train, epochs=50, batch_size=16, validation_split=0.1)

print("\n--- 4. CALCULATING THE ANOMALY THRESHOLD ---")
# Have the model predict the same healthy data it just trained on
predictions = model.predict(X_train)
# Calculate the Mean Squared Error (MSE) for every single window
train_mse = np.mean(np.square(X_train - predictions), axis=1)

# Find the maximum error it ever makes on healthy data
max_healthy_mse = np.max(train_mse)
recommended_threshold = max_healthy_mse * 1.5 # Add a 50% buffer to avoid false alarms

print(f"\n==================================================")
print(f"✅ MAX HEALTHY MSE: {max_healthy_mse:.5f}")
print(f"🚨 RECOMMENDED ESP32 ALARM THRESHOLD: {recommended_threshold:.5f}")
print(f"==================================================\n")

print("--- 5. EXPORTING TO ESP32 (.h file) ---")
# Convert the Keras model to TensorFlow Lite
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

# Format the hex array WITH LINE BREAKS so Arduino doesn't crash
hex_lines = []
for i in range(0, len(tflite_model), 12):
    chunk = tflite_model[i:i+12]
    hex_lines.append(', '.join([f'0x{b:02x}' for b in chunk]))
hex_array = ',\n    '.join(hex_lines)

c_code = f"""// Auto-generated by train_autoencoder.py
#ifndef UAV_MODEL_H
#define UAV_MODEL_H

const unsigned int uav_model_tflite_len = {len(tflite_model)};
const unsigned char uav_model_tflite[] = {{
    {hex_array}
}};

#endif
"""

# Save to a file
with open("uav_model.h", "w") as f:
    f.write(c_code)

print("✅ SUCCESS: Saved formatted 'uav_model.h'.")

--- 1. LOADING DATA ---
Generated 30 training windows of shape 300

--- 2. BUILDING THE AUTOENCODER ---


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_6 (Dense)                 │ (None, 64)             │        19,264 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 32)             │           544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 64)             │         2,112 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 300)            │        19,500 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 44,028 (171.98 KB)

 Trainable params: 44,028 (171.98 KB)

 Non-trainable params: 0 (0.00 B)


--- 3. TRAINING THE AI ---
Epoch 1/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 2s 209ms/step - loss: 2.4501 - val_loss: 2.4247
Epoch 2/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step - loss: 2.4062 - val_loss: 2.4058
Epoch 3/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step - loss: 2.3830 - val_loss: 2.3936
Epoch 4/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step - loss: 2.3618 - val_loss: 2.3815
Epoch 5/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 63ms/step - loss: 2.3339 - val_loss: 2.3688
Epoch 6/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 63ms/step - loss: 2.2971 - val_loss: 2.3554
Epoch 7/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 76ms/step - loss: 2.2475 - val_loss: 2.3413
Epoch 8/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 71ms/step - loss: 2.1851 - val_loss: 2.3245
Epoch 9/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step - loss: 2.1222 - val_loss: 2.3039
Epoch 10/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step - loss: 2.0434 - val_loss: 2.2788
Epoch 11/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step - loss: 1.9720 - val_loss: 2.2513
Epoch 12/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step - loss